In [0]:
dbutils.library.restartPython()

In [0]:
pip install serpapi

In [0]:
# DEMO CELL - DELETE AFTER TESTING
# This cell shows how to test SerpAPI connection
# Replace with your own secret setup

import serpapi

# Secure method: Use Databricks secrets
api_key = dbutils.secrets.get(scope="serpapi", key="api_key")

client = serpapi.Client(api_key=api_key)

# Test with a simple query
results = client.search({
  "engine": "google_trends",
  "q": "quantum computing",
  "date": "today 12-m",
  "tz": "420",
  "data_type": "TIMESERIES"
})

interest_over_time = results["interest_over_time"]
print("✓ SerpAPI connection successful!")

In [0]:
import serpapi
import pandas as pd
import time

# SECURE: Use Databricks secrets for API key
# Setup instructions: See README.md section "API Credentials Setup"
api_key = dbutils.secrets.get(scope="serpapi", key="api_key")

client = serpapi.Client(api_key=api_key)

# we dont need 'Jordan' within the search terms because we are restricting to searches within Jordan the country
search_terms = [
    {"search_theme": "economic_conditions", "search_term": "inflation", "language": "English"},
    {"search_theme": "economic_conditions", "search_term": "التضخم", "language": "Arabic"},
    {"search_theme": "economic_conditions", "search_term": "fuel prices", "language": "English"},
    {"search_theme": "economic_conditions", "search_term": "أسعار الوقود", "language": "Arabic"},
    {"search_theme": "economic_conditions", "search_term": "unemployment", "language": "English"},
    {"search_theme": "economic_conditions", "search_term": "البطالة", "language": "Arabic"},
    {"search_theme": "economic_conditions", "search_term": "jobs", "language": "English"},
    {"search_theme": "economic_conditions", "search_term": "وظائف", "language": "Arabic"},
    {"search_theme": "economic_conditions", "search_term": "cost of living", "language": "English"},
    {"search_theme": "economic_conditions", "search_term": "تكلفة المعيشة", "language": "Arabic"},
    {"search_theme": "civil_unrest", "search_term": "protests", "language": "English"},
    {"search_theme": "civil_unrest", "search_term": "احتجاجات", "language": "Arabic"},
    {"search_theme": "economic_conditions", "search_term": "electricity prices", "language": "English"},
    {"search_theme": "economic_conditions", "search_term": "أسعار الكهرباء", "language": "Arabic"},
    {"search_theme": "economic_conditions", "search_term": "food prices", "language": "English"},
    {"search_theme": "economic_conditions", "search_term": "أسعار المواد الغذائية", "language": "Arabic"},
    {"search_theme": "civil_unrest", "search_term": "public sector strike", "language": "English"},
    {"search_theme": "civil_unrest", "search_term": "إضراب القطاع العام", "language": "Arabic"}
]

all_rows = []

for search_config in search_terms:
    theme = search_config["search_theme"]
    term = search_config["search_term"]
    language = search_config["language"]
    
    print(f"Pulling Google Trends data for: {term} ({theme})")

    try:
        results = client.search({
            "engine": "google_trends",
            "q": term,
            "geo": "JO", # this restricts searches to be within Jordan
            "date": "today 5-y",
            "tz": "-180",
            "data_type": "TIMESERIES"
        })

        if results.get("error"):
            print(f"SerpApi error for {term}: {results['error']}")
            continue

        timeline_data = (
            results
            .get("interest_over_time", {})
            .get("timeline_data", [])
        )

        if not timeline_data:
            print(f"No Trends data returned for: {term}")
            continue

        for row in timeline_data:
            timestamp = row.get("timestamp")
            values = row.get("values", [])

            if timestamp is not None and values:
                value_info = values[0]

                all_rows.append({
                    # this is the first date of the week
                    "event_date": pd.to_datetime(int(timestamp), unit="s").date(),
                    "google_trends_date_label": row.get("date"),
                    "timestamp": int(timestamp),
                    "search_theme": theme,
                    "search_term": term,
                    "search_language": language,
                    "trend_score": value_info.get("extracted_value"),
                    "trend_value_display": value_info.get("value"),
                    "geography": "JO"
                })

    except Exception as error:
        print(f"Failed to retrieve {term}: {error}")

    time.sleep(2)

google_trends_df = pd.DataFrame(all_rows)

if google_trends_df.empty:
    raise ValueError("No Google Trends observations were returned.")

display(google_trends_df)

spark_df = spark.createDataFrame(google_trends_df)

spark.sql("CREATE SCHEMA IF NOT EXISTS info_env_jordan.bronze")

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("info_env_jordan.bronze.google_trends")

In [0]:
import serpapi
import pandas as pd
import time

# SECURE: Use Databricks secrets for API key
# Setup instructions: See README.md section "API Credentials Setup"
api_key = dbutils.secrets.get(scope="serpapi", key="api_key")

client = serpapi.Client(api_key=api_key)

# "Jordan" is not needed in the terms because geo="JO"
# restricts the Trends results to searches made within Jordan.

search_terms = [
    # --------------------------------------------------
    # Palestine solidarity
    # --------------------------------------------------
    {
        "search_theme": "palestine_solidarity",
        "search_term": "Palestine",
        "language": "English"
    },
    {
        "search_theme": "palestine_solidarity",
        "search_term": "فلسطين",
        "language": "Arabic"
    },
    {
        "search_theme": "palestine_solidarity",
        "search_term": "Palestine solidarity",
        "language": "English"
    },
    {
        "search_theme": "palestine_solidarity",
        "search_term": "التضامن مع فلسطين",
        "language": "Arabic"
    },
    {
        "search_theme": "palestine_solidarity",
        "search_term": "support Gaza",
        "language": "English"
    },
    {
        "search_theme": "palestine_solidarity",
        "search_term": "دعم غزة",
        "language": "Arabic"
    },
    {
        "search_theme": "palestine_solidarity",
        "search_term": "Palestinian cause",
        "language": "English"
    },
    {
        "search_theme": "palestine_solidarity",
        "search_term": "القضية الفلسطينية",
        "language": "Arabic"
    },

    # --------------------------------------------------
    # Gaza war and ceasefire
    # --------------------------------------------------
    {
        "search_theme": "gaza_war",
        "search_term": "Gaza",
        "language": "English"
    },
    {
        "search_theme": "gaza_war",
        "search_term": "غزة",
        "language": "Arabic"
    },
    {
        "search_theme": "gaza_war",
        "search_term": "Gaza war",
        "language": "English"
    },
    {
        "search_theme": "gaza_war",
        "search_term": "الحرب على غزة",
        "language": "Arabic"
    },
    {
        "search_theme": "gaza_ceasefire",
        "search_term": "Gaza ceasefire",
        "language": "English"
    },
    {
        "search_theme": "gaza_ceasefire",
        "search_term": "وقف إطلاق النار في غزة",
        "language": "Arabic"
    },

    # --------------------------------------------------
    # Boycott activity
    # --------------------------------------------------
    {
        "search_theme": "palestine_boycott",
        "search_term": "boycott Israel",
        "language": "English"
    },
    {
        "search_theme": "palestine_boycott",
        "search_term": "مقاطعة إسرائيل",
        "language": "Arabic"
    },
    {
        "search_theme": "palestine_boycott",
        "search_term": "boycott",
        "language": "English"
    },
    {
        "search_theme": "palestine_boycott",
        "search_term": "مقاطعة",
        "language": "Arabic"
    }
]

all_rows = []

for search_config in search_terms:
    theme = search_config["search_theme"]
    term = search_config["search_term"]
    language = search_config["language"]

    print(f"Pulling Google Trends data for: {term} ({theme})")

    try:
        results = client.search({
            "engine": "google_trends",
            "q": term,
            "geo": "JO",
            "date": "today 5-y",
            "tz": "-180",
            "data_type": "TIMESERIES"
        })

        if results.get("error"):
            print(f"SerpApi error for {term}: {results['error']}")
            continue

        timeline_data = (
            results
            .get("interest_over_time", {})
            .get("timeline_data", [])
        )

        if not timeline_data:
            print(f"No Trends data returned for: {term}")
            continue

        for row in timeline_data:
            timestamp = row.get("timestamp")
            values = row.get("values", [])

            if timestamp is not None and values:
                value_info = values[0]

                all_rows.append({
                    # For a five-year request, this will usually
                    # represent the beginning of the Trends week.
                    "event_date": pd.to_datetime(
                        int(timestamp),
                        unit="s"
                    ).date(),
                    "google_trends_date_label": row.get("date"),
                    "timestamp": int(timestamp),
                    "search_theme": theme,
                    "search_term": term,
                    "search_language": language,
                    "trend_score": value_info.get("extracted_value"),
                    "trend_value_display": value_info.get("value"),
                    "geography": "JO"
                })

    except Exception as error:
        print(f"Failed to retrieve {term}: {error}")

    time.sleep(2)

google_trends_df = pd.DataFrame(all_rows)

if google_trends_df.empty:
    raise ValueError("No Google Trends observations were returned.")

display(google_trends_df)

spark_df = spark.createDataFrame(google_trends_df)

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS info_env_jordan.bronze"
)

(
    spark_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("info_env_jordan.bronze.google_trends")
)